In [ ]:
from huggingface_hub import login
from google.colab import userdata

# This securely logs you in without exposing your password in the code
login(token=userdata.get('HF_TOKEN'))

In [ ]:
# Run this in the first Colab cell
!pip install -q transformers datasets librosa evaluate jiwer accelerate
!pip install -q indic-nlp-library # Crucial for Gujarati text cleaning

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 68.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.3/40.3 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.7/7.7 MB 76.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.1/121.1 kB 12.7 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
import os

# This will prompt you to log into your Google account
drive.mount('/content/drive')

# Create a folder in your Drive to save the model
save_directory = '/content/drive/MyDrive/Gujarati_Whisper_Project'
os.makedirs(save_directory, exist_ok=True)
print(f"Model will be saved to: {save_directory}")

Mounted at /content/drive
Model will be saved to: /content/drive/MyDrive/Gujarati_Whisper_Project


In [ ]:
# This extracts the zip file into a new folder called "my_model"
!unzip /content/drive/MyDrive/gujarati_asr_final.zip -d /content/my_model

Archive:  /content/drive/MyDrive/gujarati_asr_final.zip
  inflating: /content/my_model/config.json  
  inflating: /content/my_model/processor_config.json  
  inflating: /content/my_model/tokenizer.json  
  inflating: /content/my_model/generation_config.json  
  inflating: /content/my_model/tokenizer_config.json  
  inflating: /content/my_model/model.safetensors  


In [ ]:
import torch
from dataclasses import dataclass
from typing import Any, Dict, List, Union
from transformers import WhisperProcessor, WhisperForConditionalGeneration, Seq2SeqTrainingArguments, Seq2SeqTrainer
from datasets import load_dataset, Audio
from indicnlp.normalize.indic_normalize import IndicNormalizerFactory

# --- 1. SET UP THE GUJARATI TEXT GATE ---
factory = IndicNormalizerFactory()
normalizer = factory.get_normalizer("gu")

def clean_gujarati_text(text):
    if not isinstance(text, str): return ""
    return normalizer.normalize(text)

# --- 2. INITIALIZE PROCESSOR & MODEL ---
model_name = "openai/whisper-small"
processor = WhisperProcessor.from_pretrained(model_name, language="Gujarati", task="transcribe")
model = WhisperForConditionalGeneration.from_pretrained(model_name)

# --- 3. LOAD DATASETS (Train AND Validation) ---
# Train
train_dataset = load_dataset("ai4bharat/IndicVoices", "gujarati", split="train", streaming=True)
train_dataset = train_dataset.rename_column("audio_filepath", "audio").cast_column("audio", Audio(sampling_rate=16000))

# Validation (The missing piece!)
eval_dataset = load_dataset("ai4bharat/IndicVoices", "gujarati", split="valid", streaming=True)
eval_dataset = eval_dataset.rename_column("audio_filepath", "audio").cast_column("audio", Audio(sampling_rate=16000))

# --- 4. PREPROCESS DATA (With Length Filter) ---
def prepare_dataset(batch):
    audio = batch["audio"]
    batch["input_features"] = processor.feature_extractor(audio["array"], sampling_rate=audio["sampling_rate"]).input_features[0]
    cleaned_text = clean_gujarati_text(batch["text"])
    batch["labels"] = processor.tokenizer(cleaned_text).input_ids
    return batch

# 1. Apply the gate (clean and tokenize)
processed_train_dataset = train_dataset.map(prepare_dataset)
processed_eval_dataset = eval_dataset.map(prepare_dataset)

# 2. THE BOUNCER: Filter out anything longer than 448 tokens
MAX_TOKENS = 448

def is_in_length_range(batch):
    return len(batch["labels"]) < MAX_TOKENS

# 3. Apply the bouncer to the datasets
processed_train_dataset = processed_train_dataset.filter(is_in_length_range)
processed_eval_dataset = processed_eval_dataset.filter(is_in_length_range)

# --- 5. THE DATA COLLATOR (Padding) ---
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_features": feature["input_features"]} for feature in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        label_features = [{"input_ids": feature["labels"]} for feature in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)

# --- 6. TRAINING ARGUMENTS ---
training_args = Seq2SeqTrainingArguments(
    output_dir="/content/drive/MyDrive/gujarati_whisper_model",  # Saving locally in Colab for now
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=1e-5,
    warmup_steps=50,
    max_steps=500,
    gradient_checkpointing=True,
    fp16=True,
    eval_strategy="steps",
    predict_with_generate=True,
    generation_max_length=225,
    save_steps=100,
    eval_steps=100,
    logging_steps=25,
    report_to=["tensorboard"],
)

# --- 7. INITIALIZE THE TRAINER ---
trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=processed_train_dataset,
    eval_dataset=processed_eval_dataset,  # <-- Validation is correctly assigned here
    data_collator=data_collator,
    processing_class=processor.feature_extractor,
)

print("Pipeline compiled! Ready to train!")

CHECKPOINT_PATH = "/content/drive/MyDrive/gujarati_whisper_model/checkpoint-400"

# This tells the Trainer to load the weights and fast-forward to step 401
trainer.train(resume_from_checkpoint=CHECKPOINT_PATH)


preprocessor_config.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (1094 > 1024). Running this sequence through the model will result in indexing errors


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

RuntimeError: Could not flush decoder: Invalid data found when processing input

In [ ]:
import os
# This finds the last saved checkpoint folder (e.g., checkpoint-400 or checkpoint-500)
checkpoint_dir = [d for d in os.listdir("./gujarati_whisper_model") if "checkpoint" in d]
if checkpoint_dir:
    last_checkpoint = os.path.join("./gujarati_whisper_model", sorted(checkpoint_dir)[-1])
    print(f"Found saved progress at: {last_checkpoint}")

    # Save the final version from the last good checkpoint
    model.from_pretrained(last_checkpoint).save_pretrained("./gujarati_final_model")
    processor.save_pretrained("./gujarati_final_model")
    print("SUCCESS: Your model weights are recovered and saved to ./gujarati_final_model")
else:
    print("No checkpoints found. We may need to save the current 'model' variable directly.")
    model.save_pretrained("./gujarati_final_model")
    processor.save_pretrained("./gujarati_final_model")

Found saved progress at: ./gujarati_whisper_model/checkpoint-400


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

SUCCESS: Your model weights are recovered and saved to ./gujarati_final_model


In [ ]:
import shutil
# Zip the final model folder
shutil.make_archive("gujarati_asr_final", 'zip', "./gujarati_final_model")
# Move to Google Drive
shutil.move("gujarati_asr_final.zip", "/content/drive/MyDrive/gujarati_asr_final.zip")
print("Project Complete! Your model is safe in your Google Drive.")

Project Complete! Your model is safe in your Google Drive.
